# Experiment: Standalone Hologram Opening Example

Objective:

- import and use `hologram_opening.py` directly
- run `reconstruct_hologram(...)` in one-sideband and two-sideband modes
- use a real dataset if available, or fall back to a synthetic hologram
- inspect the returned `view_stages`, geometry, and diagnostics


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().resolve()
assert (repo_root / "hologram_opening.py").exists(), "Start Jupyter from the repository root."
sys.path.insert(0, str(repo_root))

from hologram_opening import PROCESSING_MODES, processed_phase, reconstruct_hologram
from processing import load_passage

plt.rcParams["figure.figsize"] = (15, 10)
np.set_printoptions(precision=4, suppress=True)

print(f"Using repo root: {repo_root}")
print(PROCESSING_MODES)


## Input

Leave `folder_path` empty to run the notebook end-to-end on the synthetic fallback.
Set it to a dataset folder if you want to extract a measured complex harmonic first.
Use `carrier_row_override` and `filter_width_override` to manually force the vertical filter center and width when needed.


In [ ]:
folder_path = ""
passage = "forward"
harmonic_index = 4

pad_fact = 4
alpha = 0.3
carrier_row_override = None
filter_width_override = None

folder_path = folder_path.strip()
if folder_path:
    loaded = load_passage(folder_path, passage=passage, processing_mode="two_sideband")
    source_label = f"real dataset: {loaded.image_name} ({passage}, harmonic {harmonic_index})"
    input_hologram = loaded.o[:, :, harmonic_index]
else:
    ny, nx = 160, 160
    carrier_cycles = -26
    synthetic_carrier_row_override = 56
    synthetic_filter_width_override = 24

    x = np.linspace(-1.0, 1.0, nx)[None, :]
    y = np.linspace(-1.0, 1.0, ny)[:, None]

    amplitude = 1.0 + 0.25 * np.exp(-((x + 0.25) ** 2 + (y - 0.10) ** 2) / 0.08)
    amplitude += 0.18 * np.exp(-((x - 0.30) ** 2 + (y + 0.20) ** 2) / 0.03)
    phase = 0.9 * np.exp(-((x - 0.15) ** 2 + (y + 0.10) ** 2) / 0.09)
    phase += 0.4 * np.sin(2.0 * np.pi * x) * np.exp(-(y**2) / 0.6)

    object_field = amplitude * np.exp(1j * phase)
    carrier = np.exp(1j * 2.0 * np.pi * carrier_cycles * np.arange(ny)[:, None] / ny)
    input_hologram = object_field * carrier
    source_label = "synthetic example"
    if carrier_row_override is None:
        carrier_row_override = synthetic_carrier_row_override
    if filter_width_override is None:
        filter_width_override = synthetic_filter_width_override

print({
    "source": source_label,
    "shape": input_hologram.shape,
    "pad_fact": pad_fact,
    "alpha": alpha,
    "carrier_row_override": carrier_row_override,
    "filter_width_override": filter_width_override,
})


In [ ]:
open_kwargs = {
    "pad_fact": pad_fact,
    "alpha": alpha,
}
if carrier_row_override is not None:
    open_kwargs["carrier_row"] = carrier_row_override
if filter_width_override is not None:
    open_kwargs["filter_width_y"] = filter_width_override

one_sideband_result = reconstruct_hologram(
    input_hologram,
    processing_mode="one_sideband",
    **open_kwargs,
)
two_sideband_result = reconstruct_hologram(
    input_hologram,
    processing_mode="two_sideband",
    **open_kwargs,
)

summary = {
    "one_sideband": {
        "carrier_row": one_sideband_result.geometry.carrier_row,
        "filter_width_y": one_sideband_result.geometry.filter_width_y,
        **one_sideband_result.diagnostics,
    },
    "two_sideband": {
        "carrier_row": two_sideband_result.geometry.carrier_row,
        "filter_width_y": two_sideband_result.geometry.filter_width_y,
        **two_sideband_result.diagnostics,
    },
}
summary


In [ ]:
raw_amplitude = np.abs(input_hologram)
raw_phase = np.angle(input_hologram)

one_processed = one_sideband_result.view_stages.processed
two_processed = two_sideband_result.view_stages.processed

one_amp = np.abs(one_processed)
two_amp = np.abs(two_processed)
one_phase = processed_phase(one_processed, processing_mode="one_sideband")
two_phase = processed_phase(two_processed, processing_mode="two_sideband")

fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
plots = [
    (raw_amplitude, f"Raw amplitude: {source_label}", "hot"),
    (one_amp, "Processed amplitude: one sideband", "hot"),
    (two_amp, "Processed amplitude: two sidebands", "hot"),
    (raw_phase, f"Raw phase: {source_label}", "twilight"),
    (one_phase, "Processed phase: one sideband", "bwr"),
    (two_phase, "Processed phase: two sidebands", "bwr"),
]

for axis, (image, title, cmap) in zip(axes.ravel(), plots):
    im = axis.imshow(image, aspect="auto", cmap=cmap)
    axis.set_title(title)
    axis.set_xlabel("X")
    axis.set_ylabel("Y")
    fig.colorbar(im, ax=axis, fraction=0.046, pad=0.04)

plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
stage_plots = [
    (np.log1p(one_sideband_result.view_stages.mag_signal_ft), "One-sideband log FFT magnitude", "magma"),
    (np.log1p(np.abs(one_sideband_result.view_stages.filtered_shift)), "One-sideband filtered shift magnitude", "magma"),
    (np.log1p(two_sideband_result.view_stages.mag_signal_ft), "Two-sideband log FFT magnitude", "magma"),
    (np.log1p(np.abs(two_sideband_result.view_stages.filtered_shift)), "Two-sideband filtered shift magnitude", "magma"),
]

for axis, (image, title, cmap) in zip(axes.ravel(), stage_plots):
    im = axis.imshow(image, aspect="auto", cmap=cmap)
    axis.set_title(title)
    axis.set_xlabel("kx")
    axis.set_ylabel("ky")
    fig.colorbar(im, ax=axis, fraction=0.046, pad=0.04)

plt.show()


In [ ]:
result = {
    "source": source_label,
    "one_sideband": {
        "carrier_row": one_sideband_result.geometry.carrier_row,
        "filter_width_y": one_sideband_result.geometry.filter_width_y,
        "processed_amplitude_mean": float(np.mean(one_amp)),
        "processed_phase_std": float(np.std(one_phase)),
        **one_sideband_result.diagnostics,
    },
    "two_sideband": {
        "carrier_row": two_sideband_result.geometry.carrier_row,
        "filter_width_y": two_sideband_result.geometry.filter_width_y,
        "processed_amplitude_mean": float(np.mean(two_amp)),
        "processed_phase_std": float(np.std(two_phase)),
        **two_sideband_result.diagnostics,
    },
}
result


## Next steps

- Set `folder_path` to your dataset directory if you want to run on measured data.
- Use `harmonic_index` and `passage` to choose which complex image to extract from the folder.
- If you already have a complex 2D harmonic image, the only required call is `reconstruct_hologram(...)`.
